# Feature Engineering — Hey Banco Datathon 2026

Construcción de la **tabla maestra** (una fila por usuario) que alimenta el clustering no supervisado.

**Estrategia:** foto única, todas las features estáticas. Los tópicos de Havi se pegan después en el notebook de NLP.

**Bloques que vamos a construir:**
1. Demográficos (de clientes, casi directos)
2. Producto — agregados de portafolio + tenencia binaria por tipo
3. Producto — métricas crediticias agregadas
4. Transacción — RFM (Recencia, Frecuencia, Monto)
5. Transacción — perfil de gasto por categoría MCC (% gasto)
6. Transacción — perfil de canal (% uso de cada canal)
7. Transacción — perfil de tipo de operación
8. Transacción — calidad y riesgo (% fallidas, motivos, atípicos)
9. Transacción — patrón temporal (hora, día semana, fines de semana)
10. Merge final + validaciones + export

**Naming convention:**
- `cli_*` → demográfico
- `prod_*` → portafolio de productos
- `tx_*` → comportamiento transaccional
- `flag_*` → booleanos derivados (1/0)

---
## 1. Setup y carga

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 100)
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')

DATA_DIR = '../data'

df_cli = pd.read_csv(f'{DATA_DIR}/hey_clientes.csv')
df_prod = pd.read_csv(f'{DATA_DIR}/hey_productos.csv',
                       parse_dates=['fecha_apertura', 'fecha_ultimo_movimiento'])
df_tx = pd.read_csv(f'{DATA_DIR}/hey_transacciones.csv',
                     parse_dates=['fecha_hora'])

# Eliminar columnas inútiles (siempre True)
df_prod = df_prod.drop(columns=['es_dato_sintetico'], errors='ignore')
df_tx = df_tx.drop(columns=['es_dato_sintetico'], errors='ignore')

# Fecha de corte = fecha máxima de transacciones (referencia para recencia)
FECHA_CORTE = df_tx['fecha_hora'].max()

print(f'Clientes:       {df_cli.shape}')
print(f'Productos:     {df_prod.shape}')
print(f'Transacciones: {df_tx.shape}')
print(f'Fecha de corte: {FECHA_CORTE}')
print(f'Periodo: {(FECHA_CORTE - df_tx["fecha_hora"].min()).days} días')

Clientes:       (15025, 22)
Productos:     (38909, 12)
Transacciones: (802384, 21)
Fecha de corte: 2025-12-15 13:59:36
Periodo: 348 días


---
## 2. Bloque 1 — Demográficos

Clientes ya viene a granularidad de usuario. Solo renombramos con prefijo y derivamos algunas features extra.

In [2]:
def build_features_clientes(df: pd.DataFrame) -> pd.DataFrame:
    """Demográficos directos + derivadas a nivel cliente."""
    f = df.copy()
    
    # Tratar nulls de geografía: marcar como 'desconocido' (no eliminar usuarios)
    f['estado'] = f['estado'].fillna('desconocido')
    f['ciudad'] = f['ciudad'].fillna('desconocido')
    
    # Para satisfacción NaN: indicar con flag (no es lo mismo que 'baja satisfacción')
    f['cli_satisfaccion_reportada'] = f['satisfaccion_1_10'].notna().astype(int)
    
    # Features derivadas
    f['cli_antiguedad_anios'] = f['antiguedad_dias'] / 365
    f['cli_dormido_30d'] = (f['dias_desde_ultimo_login'] > 30).astype(int)
    f['cli_dormido_60d'] = (f['dias_desde_ultimo_login'] > 60).astype(int)
    f['cli_ingreso_log'] = np.log1p(f['ingreso_mensual_mxn'])
    
    # Bandas de score de buró (estándar de industria crediticia)
    f['cli_buro_band'] = pd.cut(f['score_buro'],
                                  bins=[0, 580, 670, 740, 850],
                                  labels=['malo', 'regular', 'bueno', 'excelente'])
    
    # Bandas de edad (granularidad útil para segmentación demográfica)
    f['cli_edad_band'] = pd.cut(f['edad'],
                                  bins=[0, 25, 35, 45, 55, 100],
                                  labels=['18-25', '26-35', '36-45', '46-55', '56+'])
    
    # Rename con prefijo
    rename_map = {
        'edad': 'cli_edad', 'sexo': 'cli_sexo',
        'estado': 'cli_estado', 'ciudad': 'cli_ciudad',
        'nivel_educativo': 'cli_nivel_educativo',
        'ocupacion': 'cli_ocupacion',
        'ingreso_mensual_mxn': 'cli_ingreso_mxn',
        'antiguedad_dias': 'cli_antiguedad_dias',
        'es_hey_pro': 'cli_es_hey_pro',
        'nomina_domiciliada': 'cli_nomina_domiciliada',
        'canal_apertura': 'cli_canal_apertura',
        'score_buro': 'cli_score_buro',
        'dias_desde_ultimo_login': 'cli_dias_desde_login',
        'preferencia_canal': 'cli_preferencia_canal',
        'satisfaccion_1_10': 'cli_satisfaccion',
        'recibe_remesas': 'cli_recibe_remesas',
        'usa_hey_shop': 'cli_usa_hey_shop',
        'idioma_preferido': 'cli_idioma',
        'tiene_seguro': 'cli_tiene_seguro',
        'num_productos_activos': 'cli_num_productos_activos',
        'patron_uso_atipico': 'cli_patron_atipico',
    }
    return f.rename(columns=rename_map)

feat_cli = build_features_clientes(df_cli)
print(f'Features de clientes: {feat_cli.shape}')
print(f'Columnas: {[c for c in feat_cli.columns if c.startswith("cli_") or c == "user_id"][:10]} ...')

Features de clientes: (15025, 29)
Columnas: ['user_id', 'cli_edad', 'cli_sexo', 'cli_estado', 'cli_ciudad', 'cli_nivel_educativo', 'cli_ocupacion', 'cli_ingreso_mxn', 'cli_antiguedad_dias', 'cli_es_hey_pro'] ...


---
## 3. Bloque 2 — Productos: agregados de portafolio y tenencia

Cada usuario tiene N productos. Reducimos a una fila por usuario con:
- Conteos generales (n productos, activos, cancelados)
- Antigüedad mínima/máxima del portafolio
- **Pivot de tenencia**: una columna binaria por tipo de producto

In [3]:
CREDITO_TYPES = {
    'tarjeta_credito_hey', 'tarjeta_credito_garantizada', 'tarjeta_credito_negocios',
    'credito_personal', 'credito_auto', 'credito_nomina'
}

def build_features_productos_general(df: pd.DataFrame, fecha_corte) -> pd.DataFrame:
    """Agregados generales del portafolio del usuario."""
    df = df.copy()
    df['es_credito'] = df['tipo_producto'].isin(CREDITO_TYPES)
    df['es_activo'] = df['estatus'] == 'activo'
    df['es_cancelado'] = df['estatus'] == 'cancelado'
    df['es_suspendido'] = df['estatus'] == 'suspendido'
    df['es_revision_pagos'] = df['estatus'] == 'revision_de_pagos'
    df['dias_desde_apertura'] = (fecha_corte - df['fecha_apertura']).dt.days
    df['dias_desde_ult_mov'] = (fecha_corte - df['fecha_ultimo_movimiento']).dt.days
    
    agg = df.groupby('user_id').agg(
        prod_n_total=('producto_id', 'count'),
        prod_n_activos=('es_activo', 'sum'),
        prod_n_cancelados=('es_cancelado', 'sum'),
        prod_n_suspendidos=('es_suspendido', 'sum'),
        prod_n_revision_pagos=('es_revision_pagos', 'sum'),
        prod_n_credito=('es_credito', 'sum'),
        prod_antiguedad_max_dias=('dias_desde_apertura', 'max'),
        prod_antiguedad_min_dias=('dias_desde_apertura', 'min'),
        prod_recencia_mov_min_dias=('dias_desde_ult_mov', 'min'),
    ).reset_index()
    
    # Ratios derivados (más informativos para clustering que conteos crudos)
    agg['prod_pct_activos'] = agg['prod_n_activos'] / agg['prod_n_total']
    agg['prod_pct_cancelados'] = agg['prod_n_cancelados'] / agg['prod_n_total']
    
    # Flag: tiene algún producto en problemas
    agg['prod_flag_problemas'] = ((agg['prod_n_suspendidos'] + agg['prod_n_revision_pagos']) > 0).astype(int)
    
    return agg

feat_prod_gen = build_features_productos_general(df_prod, FECHA_CORTE)
print(f'Features generales de productos: {feat_prod_gen.shape}')
feat_prod_gen.head(3)

Features generales de productos: (15025, 13)


,user_id,prod_n_total,prod_n_activos,prod_n_cancelados,prod_n_suspendidos,prod_n_revision_pagos,prod_n_credito,prod_antiguedad_max_dias,prod_antiguedad_min_dias,prod_recencia_mov_min_dias,prod_pct_activos,prod_pct_cancelados,prod_flag_problemas
0,USR-00001,2,2,0,0,0,1,1156,903,18,1.0000,0.0000,0
1,USR-00002,2,2,0,0,0,1,256,150,60,1.0000,0.0000,0
2,USR-00003,2,2,0,0,0,1,995,600,61,1.0000,0.0000,0


In [4]:
def build_features_productos_tenencia(df: pd.DataFrame) -> pd.DataFrame:
    """Pivot binario: ¿el usuario tiene tarjeta_credito_hey, inversion_hey, etc.?"""
    # Solo contamos productos activos para 'tenencia'
    df_activos = df[df['estatus'] == 'activo']
    
    # Crosstab: filas=user, cols=tipo_producto, valores=n productos
    pivot = pd.crosstab(df_activos['user_id'], df_activos['tipo_producto'])
    
    # Renombrar con prefijo
    pivot.columns = [f'prod_tiene_{c}' for c in pivot.columns]
    
    # Convertir a binario (0/1) — para clustering la magnitud no aporta tanto
    pivot = (pivot > 0).astype(int)
    
    return pivot.reset_index()

feat_prod_tenencia = build_features_productos_tenencia(df_prod)
print(f'Features de tenencia: {feat_prod_tenencia.shape}')
feat_prod_tenencia.head(3)

Features de tenencia: (14208, 12)


,user_id,prod_tiene_credito_auto,prod_tiene_credito_nomina,prod_tiene_credito_personal,prod_tiene_cuenta_debito,prod_tiene_cuenta_negocios,prod_tiene_inversion_hey,prod_tiene_seguro_compras,prod_tiene_seguro_vida,prod_tiene_tarjeta_credito_garantizada,prod_tiene_tarjeta_credito_hey,prod_tiene_tarjeta_credito_negocios
0,USR-00001,0,0,0,1,0,0,0,0,0,1,0
1,USR-00002,0,0,0,1,0,0,0,0,0,1,0
2,USR-00003,0,0,0,1,0,0,0,0,0,1,0


---
## 4. Bloque 3 — Productos: métricas crediticias

Solo aplica a usuarios con productos de crédito. Para los demás, las features quedarán NaN (información, no problema).

In [5]:
def build_features_credito(df: pd.DataFrame) -> pd.DataFrame:
    """Agregados crediticios: límite total, utilización, tasas."""
    df_cred = df[df['tipo_producto'].isin(CREDITO_TYPES) & (df['estatus'] == 'activo')].copy()
    
    if df_cred.empty:
        return pd.DataFrame(columns=['user_id'])
    
    agg = df_cred.groupby('user_id').agg(
        cred_limite_total=('limite_credito', 'sum'),
        cred_limite_max=('limite_credito', 'max'),
        cred_saldo_total=('saldo_actual', 'sum'),
        cred_utilizacion_max=('utilizacion_pct', 'max'),
        cred_utilizacion_avg=('utilizacion_pct', 'mean'),
        cred_tasa_max=('tasa_interes_anual', 'max'),
        cred_tasa_avg=('tasa_interes_anual', 'mean'),
        cred_mensualidad_total=('monto_mensualidad', 'sum'),
    ).reset_index()
    
    # Flags útiles: alta utilización (señal de stress crediticio)
    agg['cred_flag_alta_utilizacion'] = (agg['cred_utilizacion_max'] > 0.8).astype(int)
    agg['cred_flag_critico'] = (agg['cred_utilizacion_max'] > 0.95).astype(int)
    
    return agg

feat_cred = build_features_credito(df_prod)
print(f'Features crediticios: {feat_cred.shape}')
print(f'(Solo usuarios con crédito; los demás tendrán NaN tras el merge)')
feat_cred.head(3)

Features crediticios: (9572, 11)
(Solo usuarios con crédito; los demás tendrán NaN tras el merge)


,user_id,cred_limite_total,cred_limite_max,cred_saldo_total,cred_utilizacion_max,cred_utilizacion_avg,cred_tasa_max,cred_tasa_avg,cred_mensualidad_total,cred_flag_alta_utilizacion,cred_flag_critico
0,USR-00001,"144,000.0000","144,000.0000","88,790.4000",0.6166,0.6166,35.7100,35.7100,0.0000,0,0
1,USR-00002,"22,000.0000","22,000.0000","6,122.6000",0.2783,0.2783,34.0800,34.0800,0.0000,0,0
2,USR-00003,"5,000.0000","5,000.0000","1,545.5000",0.3091,0.3091,39.8800,39.8800,0.0000,0,0


In [6]:
def build_features_inversion(df: pd.DataFrame) -> pd.DataFrame:
    """Si el usuario tiene producto de inversión: monto invertido y antigüedad."""
    df_inv = df[(df['tipo_producto'] == 'inversion_hey') & (df['estatus'] == 'activo')].copy()
    
    if df_inv.empty:
        return pd.DataFrame(columns=['user_id'])
    
    agg = df_inv.groupby('user_id').agg(
        inv_monto_total=('saldo_actual', 'sum'),
        inv_n_productos=('producto_id', 'count'),
    ).reset_index()
    
    return agg

feat_inv = build_features_inversion(df_prod)
print(f'Features de inversión: {feat_inv.shape}')
feat_inv.head(3)

Features de inversión: (4035, 3)


,user_id,inv_monto_total,inv_n_productos
0,USR-00004,"319,425.5300",1
1,USR-00007,"33,585.2900",1
2,USR-00011,"298,686.8400",1


---
## 5. Bloque 4 — Transacciones: RFM

Recencia, Frecuencia, Monto. La piedra angular del análisis comportamental.

**Importante:** solo usamos transacciones `completada` para los agregados de monto/frecuencia (las fallidas son su propio bloque).

In [7]:
def build_features_rfm(df: pd.DataFrame, fecha_corte) -> pd.DataFrame:
    """RFM clásico + extensiones útiles para banca."""
    df_ok = df[df['estatus'] == 'completada'].copy()
    
    agg = df_ok.groupby('user_id').agg(
        # Recencia
        tx_recencia_dias=('fecha_hora', lambda x: (fecha_corte - x.max()).days),
        tx_dias_activo=('fecha_hora', lambda x: (x.max() - x.min()).days),
        # Frecuencia
        tx_n_total=('transaccion_id', 'count'),
        # Monto: múltiples estadísticos para preservar la distribución
        tx_monto_total=('monto', 'sum'),
        tx_monto_avg=('monto', 'mean'),
        tx_monto_mediana=('monto', 'median'),
        tx_monto_std=('monto', 'std'),
        tx_monto_max=('monto', 'max'),
        tx_monto_p95=('monto', lambda x: x.quantile(0.95)),
    ).reset_index()
    
    # Frecuencia normalizada por días activos
    agg['tx_frec_diaria'] = agg['tx_n_total'] / agg['tx_dias_activo'].clip(lower=1)
    
    # Coeficiente de variación del monto (volatilidad relativa)
    agg['tx_monto_cv'] = agg['tx_monto_std'] / agg['tx_monto_avg'].clip(lower=1)
    
    # Log de monto total (la distribución es muy sesgada)
    agg['tx_monto_total_log'] = np.log1p(agg['tx_monto_total'])
    
    return agg

feat_rfm = build_features_rfm(df_tx, FECHA_CORTE)
print(f'Features RFM: {feat_rfm.shape}')
feat_rfm.head(3)

Features RFM: (15025, 13)


,user_id,tx_recencia_dias,tx_dias_activo,tx_n_total,tx_monto_total,tx_monto_avg,tx_monto_mediana,tx_monto_std,tx_monto_max,tx_monto_p95,tx_frec_diaria,tx_monto_cv,tx_monto_total_log
0,USR-00001,23,310,53,"118,089.4800","2,228.1034",439.0800,"5,562.4044","24,010.0000","17,938.0000",0.1710,2.4965,11.6792
1,USR-00002,18,314,71,"211,278.4800","2,975.7532",429.4600,"6,211.1743","24,830.0000","18,630.0000",0.2261,2.0873,12.2609
2,USR-00003,20,317,71,"159,295.4100","2,243.5973",559.0200,"4,653.8150","24,540.0000","12,790.0000",0.2240,2.0743,11.9785


---
## 6. Bloque 5 — Transacciones: perfil de gasto por categoría MCC

Aquí está la riqueza del comportamiento del cliente. Cada categoría MCC se vuelve una columna con el **% del gasto** del usuario en esa categoría.

**Decisión:** usamos % en vez de montos absolutos. Razón: para clustering, queremos perfilar *cómo gasta* (el mix), no *cuánto gasta* (eso ya está en RFM).

In [8]:
def build_features_mcc(df: pd.DataFrame) -> pd.DataFrame:
    """% de gasto por categoría MCC (solo compras completadas)."""
    # Solo compras: las transferencias no son 'gasto en categoría'
    df_compras = df[(df['tipo_operacion'] == 'compra') & (df['estatus'] == 'completada')].copy()
    
    # Pivot: filas=user, cols=mcc, valores=monto total
    pivot = df_compras.pivot_table(
        index='user_id', columns='categoria_mcc', values='monto', aggfunc='sum', fill_value=0
    )
    
    # Normalizar a proporciones por usuario
    pivot_pct = pivot.div(pivot.sum(axis=1), axis=0).fillna(0)
    
    # Renombrar columnas
    pivot_pct.columns = [f'tx_pct_gasto_{c}' for c in pivot_pct.columns]
    
    return pivot_pct.reset_index()

feat_mcc = build_features_mcc(df_tx)
print(f'Features MCC: {feat_mcc.shape}')
feat_mcc.head(3)

Features MCC: (14934, 14)


,user_id,tx_pct_gasto_educacion,tx_pct_gasto_entretenimiento,tx_pct_gasto_gobierno,tx_pct_gasto_hogar,tx_pct_gasto_restaurante,tx_pct_gasto_retiro_cajero,tx_pct_gasto_ropa_accesorios,tx_pct_gasto_salud,tx_pct_gasto_servicios_digitales,tx_pct_gasto_supermercado,tx_pct_gasto_tecnologia,tx_pct_gasto_transporte,tx_pct_gasto_viajes
0,USR-00001,0.0000,0.1558,0.0000,0.0000,0.4938,0.0000,0.0544,0.0000,0.2273,0.0686,0.0000,0.0000,0.0000
1,USR-00002,0.0000,0.1594,0.0000,0.0000,0.5248,0.0000,0.1506,0.0000,0.1172,0.0480,0.0000,0.0000,0.0000
2,USR-00003,0.0000,0.1587,0.0000,0.0000,0.2398,0.0000,0.1694,0.0000,0.3005,0.0335,0.0980,0.0000,0.0000


---
## 7. Bloque 6 — Transacciones: perfil de canal

% de transacciones (no monto) por canal. Distingue al cliente "todo digital" del "mixto cajero".

In [9]:
def build_features_canal(df: pd.DataFrame) -> pd.DataFrame:
    """% de transacciones por canal (todas las completadas)."""
    df_ok = df[df['estatus'] == 'completada'].copy()
    
    pivot = pd.crosstab(df_ok['user_id'], df_ok['canal'])
    pivot_pct = pivot.div(pivot.sum(axis=1), axis=0).fillna(0)
    pivot_pct.columns = [f'tx_pct_canal_{c}' for c in pivot_pct.columns]
    
    # Feature derivada: % en cualquier app vs canales físicos (engagement digital)
    app_cols = [c for c in pivot_pct.columns if 'app_' in c]
    pivot_pct['tx_pct_canal_app_total'] = pivot_pct[app_cols].sum(axis=1)
    
    return pivot_pct.reset_index()

feat_canal = build_features_canal(df_tx)
print(f'Features canal: {feat_canal.shape}')
feat_canal.head(3)

Features canal: (15025, 11)


,user_id,tx_pct_canal_app_android,tx_pct_canal_app_huawei,tx_pct_canal_app_ios,tx_pct_canal_cajero_banregio,tx_pct_canal_cajero_externo,tx_pct_canal_codi,tx_pct_canal_farmacia_ahorro,tx_pct_canal_oxxo,tx_pct_canal_pos_fisico,tx_pct_canal_app_total
0,USR-00001,0.4906,0.0377,0.2642,0.0000,0.0000,0.0377,0.0000,0.0377,0.1321,0.7925
1,USR-00002,0.3521,0.0563,0.2958,0.0282,0.0000,0.0141,0.0000,0.0000,0.2535,0.7042
2,USR-00003,0.2958,0.0563,0.2817,0.0141,0.0141,0.0141,0.0000,0.0282,0.2958,0.6338


---
## 8. Bloque 7 — Transacciones: perfil por tipo de operación

In [10]:
def build_features_tipo_op(df: pd.DataFrame) -> pd.DataFrame:
    """% de transacciones por tipo de operación."""
    df_ok = df[df['estatus'] == 'completada'].copy()
    
    pivot = pd.crosstab(df_ok['user_id'], df_ok['tipo_operacion'])
    pivot_pct = pivot.div(pivot.sum(axis=1), axis=0).fillna(0)
    pivot_pct.columns = [f'tx_pct_op_{c}' for c in pivot_pct.columns]
    
    return pivot_pct.reset_index()

feat_tipo_op = build_features_tipo_op(df_tx)
print(f'Features tipo operación: {feat_tipo_op.shape}')
feat_tipo_op.head(3)

Features tipo operación: (15025, 13)


,user_id,tx_pct_op_abono_inversion,tx_pct_op_cargo_recurrente,tx_pct_op_cashback,tx_pct_op_compra,tx_pct_op_deposito_farmacia,tx_pct_op_deposito_oxxo,tx_pct_op_pago_credito,tx_pct_op_pago_servicio,tx_pct_op_retiro_cajero,tx_pct_op_retiro_inversion,tx_pct_op_transf_entrada,tx_pct_op_transf_salida
0,USR-00001,0.0000,0.1509,0.0566,0.5472,0.0000,0.0377,0.0377,0.0377,0.0000,0.0000,0.0377,0.0943
1,USR-00002,0.0000,0.1690,0.0423,0.5634,0.0000,0.0000,0.0423,0.0000,0.0282,0.0000,0.0845,0.0704
2,USR-00003,0.0000,0.1127,0.0423,0.5493,0.0000,0.0282,0.0282,0.0704,0.0282,0.0000,0.0704,0.0704


---
## 9. Bloque 8 — Transacciones: calidad y riesgo

Aquí está la señal de fricción y posible churn: % transacciones fallidas, motivos dominantes, atípicos.

**Esto es oro para los triggers de negocio del pitch.**

In [11]:
def build_features_calidad(df: pd.DataFrame) -> pd.DataFrame:
    """% de transacciones fallidas, en disputa, atípicas, etc."""
    df = df.copy()
    df['flag_no_proc'] = (df['estatus'] == 'no_procesada').astype(int)
    df['flag_disputa'] = (df['estatus'] == 'en_disputa').astype(int)
    df['flag_revertida'] = (df['estatus'] == 'revertida').astype(int)
    df['flag_atipico'] = df['patron_uso_atipico'].astype(int)
    df['flag_internacional'] = df['es_internacional'].astype(int)
    df['flag_reintento'] = (df['intento_numero'] > 1).astype(int)
    
    agg = df.groupby('user_id').agg(
        tx_pct_no_procesada=('flag_no_proc', 'mean'),
        tx_pct_disputa=('flag_disputa', 'mean'),
        tx_pct_revertida=('flag_revertida', 'mean'),
        tx_pct_atipico=('flag_atipico', 'mean'),
        tx_pct_internacional=('flag_internacional', 'mean'),
        tx_pct_reintento=('flag_reintento', 'mean'),
        tx_n_no_procesada=('flag_no_proc', 'sum'),
    ).reset_index()
    
    return agg

feat_calidad = build_features_calidad(df_tx)
print(f'Features calidad: {feat_calidad.shape}')
feat_calidad.head(3)

Features calidad: (15025, 8)


,user_id,tx_pct_no_procesada,tx_pct_disputa,tx_pct_revertida,tx_pct_atipico,tx_pct_internacional,tx_pct_reintento,tx_n_no_procesada
0,USR-00001,0.0536,0.0000,0.0000,0.0000,0.0357,0.0357,3
1,USR-00002,0.0135,0.0135,0.0135,0.0000,0.0135,0.0135,1
2,USR-00003,0.0506,0.0380,0.0127,0.0000,0.0506,0.0127,4


In [12]:
def build_features_motivos_falla(df: pd.DataFrame) -> pd.DataFrame:
    """% de cada motivo de no procesamiento sobre el total de fallidas.
    
    Si un usuario tiene muchas 'limite_excedido' es señal de cross-sell de crédito.
    Si tiene muchas 'tarjeta_bloqueada' es señal de fricción de seguridad.
    """
    df_fail = df[df['estatus'] == 'no_procesada'].copy()
    
    if df_fail.empty:
        return pd.DataFrame(columns=['user_id'])
    
    pivot = pd.crosstab(df_fail['user_id'], df_fail['motivo_no_procesada'])
    # Normalizar dentro de fallidas (proporción del motivo dentro de las fallidas del user)
    pivot_pct = pivot.div(pivot.sum(axis=1), axis=0).fillna(0)
    pivot_pct.columns = [f'tx_motivo_{c}' for c in pivot_pct.columns]
    
    return pivot_pct.reset_index()

feat_motivos = build_features_motivos_falla(df_tx)
print(f'Features motivos de falla: {feat_motivos.shape}')
feat_motivos.head(3)

Features motivos de falla: (11098, 9)


,user_id,tx_motivo_codigo_incorrecto,tx_motivo_cuenta_destino_invalida,tx_motivo_datos_invalidos,tx_motivo_limite_excedido,tx_motivo_monto_excede_limite_diario,tx_motivo_saldo_insuficiente,tx_motivo_tarjeta_bloqueada,tx_motivo_timeout_banco
0,USR-00001,0.0000,0.3333,0.3333,0.0000,0.0000,0.3333,0.0000,0.0000
1,USR-00002,0.0000,0.0000,0.0000,0.0000,0.0000,1.0000,0.0000,0.0000
2,USR-00003,0.2500,0.0000,0.0000,0.0000,0.2500,0.5000,0.0000,0.0000


In [13]:
def build_features_cashback(df: pd.DataFrame) -> pd.DataFrame:
    """Cashback acumulado y MSI (relevante para Hey Pro)."""
    df = df.copy()
    df['flag_msi'] = df['meses_diferidos'].notna().astype(int)
    
    agg = df.groupby('user_id').agg(
        tx_cashback_total=('cashback_generado', 'sum'),
        tx_n_msi=('flag_msi', 'sum'),
        tx_pct_msi=('flag_msi', 'mean'),
    ).reset_index()
    
    agg['tx_cashback_total'] = agg['tx_cashback_total'].fillna(0)
    return agg

feat_cashback = build_features_cashback(df_tx)
print(f'Features cashback/MSI: {feat_cashback.shape}')
feat_cashback.head(3)

Features cashback/MSI: (15025, 4)


,user_id,tx_cashback_total,tx_n_msi,tx_pct_msi
0,USR-00001,122.3300,0,0.0000
1,USR-00002,147.0800,0,0.0000
2,USR-00003,165.4300,0,0.0000


---
## 10. Bloque 9 — Transacciones: patrón temporal

Cuándo transacciona el usuario. Distingue al "diurno laboral" del "nocturno fin de semana".

In [14]:
def build_features_temporal(df: pd.DataFrame) -> pd.DataFrame:
    """Patrones de hora y día de la semana."""
    df = df.copy()
    df['flag_finde'] = df['dia_semana'].isin(['Saturday', 'Sunday']).astype(int)
    df['flag_madrugada'] = ((df['hora_del_dia'] >= 0) & (df['hora_del_dia'] < 6)).astype(int)
    df['flag_horario_laboral'] = ((df['hora_del_dia'] >= 9) & (df['hora_del_dia'] < 18)).astype(int)
    df['flag_nocturno'] = ((df['hora_del_dia'] >= 22) | (df['hora_del_dia'] < 1)).astype(int)
    
    df_ok = df[df['estatus'] == 'completada']
    
    agg = df_ok.groupby('user_id').agg(
        tx_pct_finde=('flag_finde', 'mean'),
        tx_pct_madrugada=('flag_madrugada', 'mean'),
        tx_pct_horario_laboral=('flag_horario_laboral', 'mean'),
        tx_pct_nocturno=('flag_nocturno', 'mean'),
        tx_hora_modal=('hora_del_dia', lambda x: x.mode().iloc[0] if len(x.mode()) > 0 else np.nan),
        tx_hora_avg=('hora_del_dia', 'mean'),
    ).reset_index()
    
    return agg

feat_temporal = build_features_temporal(df_tx)
print(f'Features temporales: {feat_temporal.shape}')
feat_temporal.head(3)

Features temporales: (15025, 7)


,user_id,tx_pct_finde,tx_pct_madrugada,tx_pct_horario_laboral,tx_pct_nocturno,tx_hora_modal,tx_hora_avg
0,USR-00001,0.3585,0.2830,0.4906,0.0566,1,9.9245
1,USR-00002,0.2113,0.1690,0.3662,0.1408,19,13.1268
2,USR-00003,0.2254,0.1831,0.4789,0.0563,17,11.9437


---
## 11. Merge final + validaciones

In [15]:
# Partimos siempre de clientes (15,025 filas) y hacemos LEFT MERGE de todo
# Esto garantiza que el resultado tiene exactamente 15,025 filas (1 por cliente)

master = feat_cli.copy()
print(f'Inicio (clientes): {master.shape}')

for nombre, df_feat in [
    ('productos_general', feat_prod_gen),
    ('productos_tenencia', feat_prod_tenencia),
    ('credito', feat_cred),
    ('inversion', feat_inv),
    ('rfm', feat_rfm),
    ('mcc', feat_mcc),
    ('canal', feat_canal),
    ('tipo_op', feat_tipo_op),
    ('calidad', feat_calidad),
    ('motivos', feat_motivos),
    ('cashback', feat_cashback),
    ('temporal', feat_temporal),
]:
    n_antes = len(master)
    master = master.merge(df_feat, on='user_id', how='left')
    assert len(master) == n_antes, f'❌ Merge {nombre} infló filas'
    print(f'+ {nombre:25s}: {df_feat.shape[1]-1:3d} cols nuevas → master = {master.shape}')

print(f'\n✅ Master features: {master.shape}')

Inicio (clientes): (15025, 29)
+ productos_general        :  12 cols nuevas → master = (15025, 41)
+ productos_tenencia       :  11 cols nuevas → master = (15025, 52)
+ credito                  :  10 cols nuevas → master = (15025, 62)
+ inversion                :   2 cols nuevas → master = (15025, 64)
+ rfm                      :  12 cols nuevas → master = (15025, 76)
+ mcc                      :  13 cols nuevas → master = (15025, 89)
+ canal                    :  10 cols nuevas → master = (15025, 99)
+ tipo_op                  :  12 cols nuevas → master = (15025, 111)
+ calidad                  :   7 cols nuevas → master = (15025, 118)
+ motivos                  :   8 cols nuevas → master = (15025, 126)
+ cashback                 :   3 cols nuevas → master = (15025, 129)
+ temporal                 :   6 cols nuevas → master = (15025, 135)

✅ Master features: (15025, 135)


In [16]:
# Validaciones finales
print('=== Validaciones ===\n')

# 1. Una fila por usuario
assert master['user_id'].is_unique, '❌ user_id no es único'
print(f'✅ user_id único: {master["user_id"].nunique():,} usuarios')

# 2. Coincide con clientes original
assert len(master) == len(df_cli), '❌ Perdimos o ganamos filas'
print(f'✅ Mismas filas que clientes: {len(master):,}')

# 3. Resumen de tipos
print(f'\nTipos de features:')
for prefix in ['cli_', 'prod_', 'cred_', 'inv_', 'tx_']:
    cols = [c for c in master.columns if c.startswith(prefix)]
    print(f'  {prefix:10s}: {len(cols):3d} columnas')

# 4. Nulls esperados (no problemáticos)
print(f'\nNulls por bloque (esperados — no se imputan a 0):')
for prefix in ['cred_', 'inv_', 'tx_motivo_']:
    cols = [c for c in master.columns if c.startswith(prefix)]
    if cols:
        n_null_avg = master[cols].isna().sum().mean()
        print(f'  {prefix:15s}: {n_null_avg:.0f} nulls promedio (esperado: usuarios sin esa señal)')

=== Validaciones ===

✅ user_id único: 15,025 usuarios
✅ Mismas filas que clientes: 15,025

Tipos de features:
  cli_      :  28 columnas
  prod_     :  23 columnas
  cred_     :  10 columnas
  inv_      :   2 columnas
  tx_       :  71 columnas

Nulls por bloque (esperados — no se imputan a 0):
  cred_          : 5453 nulls promedio (esperado: usuarios sin esa señal)
  inv_           : 10990 nulls promedio (esperado: usuarios sin esa señal)
  tx_motivo_     : 3927 nulls promedio (esperado: usuarios sin esa señal)


In [17]:
# Vista panorámica
print(f'Total columnas: {master.shape[1]}')
print(f'\nMuestra de las primeras 3 filas con columnas seleccionadas:')
cols_sample = (['user_id', 'cli_edad', 'cli_es_hey_pro', 'prod_n_total', 'tx_n_total', 
                'tx_monto_total', 'tx_pct_no_procesada', 'tx_pct_canal_app_total'])
master[cols_sample].head(3)

Total columnas: 135

Muestra de las primeras 3 filas con columnas seleccionadas:


,user_id,cli_edad,cli_es_hey_pro,prod_n_total,tx_n_total,tx_monto_total,tx_pct_no_procesada,tx_pct_canal_app_total
0,USR-00001,21,True,2,53,"118,089.4800",0.0536,0.7925
1,USR-00002,18,True,2,71,"211,278.4800",0.0135,0.7042
2,USR-00003,23,True,2,71,"159,295.4100",0.0506,0.6338


---
## 12. Tratamiento de NaN antes de clustering

**No imputamos por default.** Cada bloque tiene una semántica distinta del NaN:
- `cred_*` NaN → usuario sin productos de crédito
- `inv_*` NaN → usuario sin inversión
- `tx_motivo_*` NaN → usuario sin transacciones fallidas

Para clustering hay que decidir qué hacer. La estrategia recomendada:

In [18]:
def preparar_para_clustering(master: pd.DataFrame) -> pd.DataFrame:
    """
    Imputación semánticamente correcta para clustering:
    - Conteos/totales: NaN → 0 (no tener productos de crédito = 0 utilización)
    - Porcentajes/proporciones: NaN → 0 (no usó ese motivo = 0% del total)
    - Tasas/medias: NaN → 0 con flag adicional 'no aplica'
    """
    df = master.copy()
    
    # Crear flags de 'tiene crédito' y 'tiene inversión' antes de imputar
    df['flag_tiene_credito'] = df['cred_limite_total'].notna().astype(int)
    df['flag_tiene_inversion'] = df['inv_monto_total'].notna().astype(int)
    df['flag_tuvo_falla'] = df['tx_n_no_procesada'].fillna(0).gt(0).astype(int)
    
    # Imputar a 0 todas las columnas numéricas con NaN (todas tienen semántica 'no aplica = 0')
    num_cols = df.select_dtypes(include=[np.number]).columns
    df[num_cols] = df[num_cols].fillna(0)
    
    # Categóricas con NaN: por si acaso
    cat_cols = df.select_dtypes(include=['object', 'category']).columns
    for c in cat_cols:
        df[c] = df[c].fillna('desconocido')
    
    return df

master_clean = preparar_para_clustering(master)
print(f'Master listo para clustering: {master_clean.shape}')
print(f'Nulls restantes: {master_clean.isna().sum().sum()} (debe ser 0)')

Master listo para clustering: (15025, 138)
Nulls restantes: 0 (debe ser 0)


---
## 13. Export

Guardamos en parquet (más rápido y preserva tipos). Persona 2 lee este archivo desde el notebook de NLP/clustering.

In [20]:
# Versión con NaN preservados (útil para análisis exploratorio)
master.to_csv(f'{DATA_DIR}/master_features_raw.csv', index=False)

# Versión imputada lista para clustering
master_clean.to_csv(f'{DATA_DIR}/master_features_clean.csv', index=False)

print('Exportados:')
print(f'  ../data/master_features_raw.csv   ({master.shape})')
print(f'  ../data/master_features_clean.csv ({master_clean.shape})')

# Catálogo de features para documentación
catalogo = pd.DataFrame({
    'feature': master_clean.columns,
    'tipo': master_clean.dtypes.astype(str).values,
    'bloque': [c.split('_')[0] if '_' in c else 'meta' for c in master_clean.columns],
})
catalogo.to_csv(f'{DATA_DIR}/catalogo_features.csv', index=False)
print(f'  ../data/catalogo_features.csv         ({len(catalogo)} features documentadas)')

Exportados:
  ../data/master_features_raw.csv   ((15025, 135))
  ../data/master_features_clean.csv ((15025, 138))
  ../data/catalogo_features.csv         (138 features documentadas)
